# Phase 5: Executive Dashboard & API Layer
## REST API Design, Compliance Visualization, and Cognito Authentication

**Time Estimate:** 8–10 hours | **Prerequisites:** Phases 2–4 completed

---

### What you'll build
A REST API that serves compliance data (assessments, posture, drift events) and a dashboard that visualizes it — using the same `ControlAssessment` and `CompliancePosture` objects from previous phases.

### Study cross-references
| Concept | DDIA Chapter | DVA-C02 | System Design Interview |
|---------|-------------|---------|-------------------------|
| Derived data & caching | Ch. 12: The Future of Data Systems | API Gateway caching | Ch. 1: Rate Limiter |
| API maintainability | Ch. 1: Maintainability | API Gateway versioning | Ch. 12: Chat System |
| Request routing | Ch. 6: Partitioning | API Gateway routing | Ch. 5: Consistent Hashing |
| Event-driven updates | Ch. 11: Stream Processing | DynamoDB Streams → Lambda | Ch. 11: News Feed |

### Documentation links
- [API Gateway REST API](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-rest-api.html)
- [API Gateway HTTP API](https://docs.aws.amazon.com/apigateway/latest/developerguide/http-api.html)
- [Amazon Cognito User Pools](https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools.html)
- [Cognito JWT Tokens](https://docs.aws.amazon.com/cognito/latest/developerguide/amazon-cognito-user-pools-using-tokens-verifying-a-jwt.html)
- [Streamlit Documentation](https://docs.streamlit.io/)
- [Plotly Python](https://plotly.com/python/)

---

## Part 1: REST API Design Theory

### 1.1 Resource-Oriented Design

REST APIs expose **resources** (nouns) with standard HTTP methods (verbs). Our compliance data maps naturally:

```
GET  /scans                         → List scan history
GET  /scans/{scan_id}/assessments   → All ControlAssessments for a scan
GET  /scans/{scan_id}/posture       → CompliancePosture for a scan
GET  /controls/{control_id}/history → Assessment history across scans
GET  /drift?since={scan_id}         → DriftEvents between scans
POST /reports                       → Trigger PDF generation (Phase 3)
```

Each endpoint returns JSON serialized from our `src/models` classes — `ControlAssessment.to_dict()`, `CompliancePosture.to_dict()`, `DriftEvent.to_dict()`.

### 1.2 DDIA Connection: Derived Data Serving (Ch. 12)

Kleppmann's key insight (p. 504): *"The derived dataset is where you get the best of both worlds: the advantages of event logs for input, and the advantages of databases for serving."*

Our API serves **derived data**:
- Raw input: `ScanResult` (event log from collectors)
- Derived: `ControlAssessment` (computed by mapping engine)
- Derived: `CompliancePosture` (aggregated from assessments)
- Derived: `DriftEvent` (computed by comparing two scans)

The dashboard never queries raw findings. It only reads the derived views — exactly like a materialized view in a database.

### 1.3 DVA-C02: API Gateway — REST vs HTTP API

| Feature | REST API | HTTP API |
|---------|----------|----------|
| Cost | $3.50/M requests | $0.90/M requests |
| Caching | Built-in (CloudFront) | No |
| Auth | IAM, Cognito, Lambda auth | JWT (Cognito) |
| Latency | ~350ms | ~200ms |
| Use for compliance? | **Yes** — caching for posture | Maybe — if cost-sensitive |

**Exam pattern:** "Which API type supports response caching?" → REST API (not HTTP API).

---

## Part 2: Amazon Cognito Authentication

### DVA-C02 Connection (Domain 3 — Security)

Cognito secures the API so only authorized users see compliance data.

**Authentication flow:**
```
1. User signs in with email/password
2. Cognito validates → returns JWT token (id_token, access_token, refresh_token)
3. Client includes JWT: Authorization: Bearer <token>
4. API Gateway authorizer validates JWT signature with Cognito public key
5. Lambda receives user claims: event['requestContext']['authorizer']['claims']
```

**JWT structure (exam-relevant):**
```json
{
  "sub": "user-uuid",
  "iss": "https://cognito-idp.us-east-1.amazonaws.com/us-east-1_xxxx",
  "aud": "client_id",
  "exp": 1234567890,
  "email": "jon@example.com",
  "custom:role": "auditor"
}
```

**Key exam facts:**
- `id_token` contains user attributes (email, name, custom claims)
- `access_token` controls API access (scopes)
- `refresh_token` is long-lived (30 days default), gets new access tokens
- MFA configuration: OFF, OPTIONAL, REQUIRED
- Password policy: min length, uppercase, lowercase, numbers, symbols

---

## Lab 5.1: Building Dashboard Data from the Pipeline

We start by running the same pipeline as Phases 2–3 to produce real data. The dashboard will visualize these exact objects.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from src.models import (
    EvidenceItem, ScanResult, CollectorResult, ControlAssessment,
    ControlStatus, CompliancePosture, DriftEvent,
    generate_scan_id, generate_drift_id,
    SEVERITY_WEIGHTS, CONTROL_FAMILIES
)
from src.mapper.control_catalog import NIST_CONTROL_CATALOG, get_control
from src.mapper.engine import ControlMappingEngine
from src.drift.detector import DriftDetector

print(f"Loaded src/ modules:")
print(f"  Models: ControlAssessment, CompliancePosture, DriftEvent")
print(f"  Engine: ControlMappingEngine")
print(f"  Drift: DriftDetector")
print(f"  Catalog: {len(NIST_CONTROL_CATALOG)} controls")

In [ ]:
# Build two scans to have both posture AND drift data for the dashboard
# Scan 1: baseline (some failures)
evidence_scan1 = [
    EvidenceItem(source="security_hub", finding_id="sh-iam4-001",
        title="IAM.4 Root MFA not enabled", status="FAILED", severity="CRITICAL",
        resource_type="AWS::IAM::User", resource_id="arn:aws:iam::123456789012:root",
        timestamp="2024-01-08T10:00:00Z", remediation="Enable hardware MFA on root",
        control_ids=["AC-2", "IA-2", "IA-2(1)"]),
    EvidenceItem(source="security_hub", finding_id="sh-s3-005",
        title="S3.5 Buckets should require SSL", status="FAILED", severity="HIGH",
        resource_type="AWS::S3::Bucket", resource_id="arn:aws:s3:::prod-data",
        timestamp="2024-01-08T10:01:00Z", remediation="Add SecureTransport policy",
        control_ids=["SC-8", "SC-13"]),
    EvidenceItem(source="config", finding_id="cfg-ct-001",
        title="multi-region-cloudtrail-enabled: COMPLIANT", status="PASSED",
        severity="INFORMATIONAL", resource_type="AWS::CloudTrail::Trail",
        resource_id="arn:aws:cloudtrail:us-east-1:trail/org-trail",
        timestamp="2024-01-08T10:02:00Z", control_ids=["AU-2", "AU-3", "AU-12"]),
    EvidenceItem(source="config", finding_id="cfg-gd-001",
        title="guardduty-enabled: COMPLIANT", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::GuardDuty::Detector", resource_id="detector-us-east-1",
        timestamp="2024-01-08T10:03:00Z", control_ids=["SI-4"]),
    EvidenceItem(source="config", finding_id="cfg-ssh-001",
        title="restricted-ssh: COMPLIANT", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::SecurityGroup", resource_id="sg-0abc123",
        timestamp="2024-01-08T10:04:00Z", control_ids=["SC-7", "CM-6"]),
    EvidenceItem(source="security_hub", finding_id="sh-iam1-001",
        title="IAM.1 No wildcard admin policies", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::IAM::Policy", resource_id="arn:aws:iam::123456789012:policy/Dev",
        timestamp="2024-01-08T10:05:00Z", control_ids=["AC-6", "AC-3"]),
]

# Scan 2: one week later (root MFA fixed, but new EBS issue)
evidence_scan2 = [
    EvidenceItem(source="security_hub", finding_id="sh-iam4-001",
        title="IAM.4 Root MFA enabled", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::IAM::User", resource_id="arn:aws:iam::123456789012:root",
        timestamp="2024-01-15T10:00:00Z",
        control_ids=["AC-2", "IA-2", "IA-2(1)"]),
    EvidenceItem(source="security_hub", finding_id="sh-s3-005",
        title="S3.5 Buckets should require SSL", status="FAILED", severity="HIGH",
        resource_type="AWS::S3::Bucket", resource_id="arn:aws:s3:::prod-data",
        timestamp="2024-01-15T10:01:00Z", remediation="Add SecureTransport policy",
        control_ids=["SC-8", "SC-13"]),
    EvidenceItem(source="config", finding_id="cfg-ct-001",
        title="multi-region-cloudtrail-enabled: COMPLIANT", status="PASSED",
        severity="INFORMATIONAL", resource_type="AWS::CloudTrail::Trail",
        resource_id="arn:aws:cloudtrail:us-east-1:trail/org-trail",
        timestamp="2024-01-15T10:02:00Z", control_ids=["AU-2", "AU-3", "AU-12"]),
    EvidenceItem(source="config", finding_id="cfg-gd-001",
        title="guardduty-enabled: COMPLIANT", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::GuardDuty::Detector", resource_id="detector-us-east-1",
        timestamp="2024-01-15T10:03:00Z", control_ids=["SI-4"]),
    EvidenceItem(source="config", finding_id="cfg-ssh-001",
        title="restricted-ssh: COMPLIANT", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::EC2::SecurityGroup", resource_id="sg-0abc123",
        timestamp="2024-01-15T10:04:00Z", control_ids=["SC-7", "CM-6"]),
    EvidenceItem(source="security_hub", finding_id="sh-iam1-001",
        title="IAM.1 No wildcard admin policies", status="PASSED", severity="INFORMATIONAL",
        resource_type="AWS::IAM::Policy", resource_id="arn:aws:iam::123456789012:policy/Dev",
        timestamp="2024-01-15T10:05:00Z", control_ids=["AC-6", "AC-3"]),
    EvidenceItem(source="config", finding_id="cfg-ebs-001",
        title="encrypted-volumes: NON_COMPLIANT", status="FAILED", severity="HIGH",
        resource_type="AWS::EC2::Volume", resource_id="vol-0abc123",
        timestamp="2024-01-15T10:06:00Z", remediation="Enable EBS default encryption",
        control_ids=["SC-13", "SC-28"]),
]

# Run both through the pipeline
engine = ControlMappingEngine()

scan1 = ScanResult(scan_id="2024-01-08T10-00-00Z_baseline", scan_start="2024-01-08T10:00:00Z",
                   account_id="123456789012", region="us-east-1")
cr1 = CollectorResult(source="collectors", status="SUCCESS", evidence_items=evidence_scan1,
                      raw_findings_count=len(evidence_scan1))
scan1.collector_results["all"] = cr1
scan1.finalize()
assessments1 = engine.assess_all_controls(scan1)
posture1 = engine.generate_posture(assessments1)

scan2 = ScanResult(scan_id="2024-01-15T10-00-00Z_weekly", scan_start="2024-01-15T10:00:00Z",
                   account_id="123456789012", region="us-east-1")
cr2 = CollectorResult(source="collectors", status="SUCCESS", evidence_items=evidence_scan2,
                      raw_findings_count=len(evidence_scan2))
scan2.collector_results["all"] = cr2
scan2.finalize()
assessments2 = engine.assess_all_controls(scan2)
posture2 = engine.generate_posture(assessments2)

# Detect drift
detector = DriftDetector()
drift_events = detector.detect(assessments1, assessments2)

print(f"Scan 1: {posture1.compliance_percentage:.1f}% compliance ({posture1.passed} pass, {posture1.failed} fail)")
print(f"Scan 2: {posture2.compliance_percentage:.1f}% compliance ({posture2.passed} pass, {posture2.failed} fail)")
print(f"Drift events: {len(drift_events)}")
for d in drift_events:
    print(f"  {d.control_id}: {d.previous_status} -> {d.current_status} ({d.drift_type})")

## Lab 5.2: Designing the Lambda API Handler

The Lambda handler serves assessment data as JSON. Each endpoint serializes `src/models` objects using their `.to_dict()` methods — the same serialization that DynamoDB uses.

**Key design decision (DDIA Ch. 12):** The API serves pre-computed derived data. It does NOT run the mapping engine on every request. Assessments are computed once (Phase 2), stored in DynamoDB, and served as-is. This is the "pre-materialized view" pattern.

```
Scan completes → Lambda runs engine → Stores ControlAssessment items in DynamoDB
                                    → Stores CompliancePosture in DynamoDB
Dashboard request → Lambda reads from DynamoDB → Returns JSON
```

In [ ]:
import json
from datetime import datetime, timezone

# Simulate the API responses using our REAL pipeline data
# In production, these come from DynamoDB; here we use in-memory data

def api_get_posture(posture: CompliancePosture) -> dict:
    """GET /scans/{scan_id}/posture -> CompliancePosture as JSON."""
    return {
        'statusCode': 200,
        'headers': {'Content-Type': 'application/json', 'Access-Control-Allow-Origin': '*'},
        'body': json.dumps(posture.to_dict(), default=str)
    }

def api_get_assessments(assessments: list, family: str = None, status: str = None) -> dict:
    """GET /scans/{scan_id}/assessments?family=AC&status=FAIL -> filtered list."""
    filtered = assessments
    if family:
        filtered = [a for a in filtered if a.control_family == family]
    if status:
        filtered = [a for a in filtered if a.status.value == status]
    
    return {
        'statusCode': 200,
        'headers': {'Content-Type': 'application/json'},
        'body': json.dumps({
            'assessments': [a.to_dict() for a in filtered],
            'count': len(filtered),
            'filters': {'family': family, 'status': status}
        }, default=str)
    }

def api_get_drift(drift_events: list) -> dict:
    """GET /drift?since={scan_id} -> DriftEvent list."""
    return {
        'statusCode': 200,
        'headers': {'Content-Type': 'application/json'},
        'body': json.dumps({
            'drift_events': [d.to_dict() for d in drift_events],
            'regressions': sum(1 for d in drift_events if d.is_regression),
            'improvements': sum(1 for d in drift_events if d.drift_type == 'IMPROVEMENT'),
        }, default=str)
    }

# Test the API responses with our real data
print("=== GET /posture ===")
posture_resp = api_get_posture(posture2)
posture_body = json.loads(posture_resp['body'])
print(f"  compliance_percentage: {posture_body['compliance_percentage']}%")
print(f"  passed: {posture_body['passed']}, failed: {posture_body['failed']}")
print(f"  families: {list(posture_body['by_family'].keys())}")

print("\n=== GET /assessments?status=FAIL ===")
fail_resp = api_get_assessments(assessments2, status="FAIL")
fail_body = json.loads(fail_resp['body'])
print(f"  {fail_body['count']} failed controls:")
for a in fail_body['assessments'][:5]:
    print(f"    {a['control_id']}: {a['control_title']} (priority {a['remediation_priority']})")

print("\n=== GET /drift ===")
drift_resp = api_get_drift(drift_events)
drift_body = json.loads(drift_resp['body'])
print(f"  {len(drift_body['drift_events'])} events: "
      f"{drift_body['regressions']} regressions, {drift_body['improvements']} improvements")

### Exercise 5.1: Build the Full Lambda Handler

Write a `lambda_handler(event, context)` function that:
1. Routes based on `event['httpMethod']` and `event['path']`
2. Extracts user info from `event['requestContext']['authorizer']['claims']`
3. Handles these routes: `GET /posture`, `GET /assessments`, `GET /drift`, `POST /reports`
4. Returns proper error responses (404 for unknown paths, 400 for missing params)
5. Uses `ControlAssessment.from_dict()` to deserialize from DynamoDB

**Hint:** Look at how `api_get_posture()` above structures the response. The handler just dispatches to the right function based on the path.

**DVA-C02 exam pattern:** "A Lambda function integrated with API Gateway needs to return a response. What format must the response use?" → `{'statusCode': 200, 'headers': {...}, 'body': '<JSON string>'}`. The body must be a string, not a dict.

---

## Lab 5.3: Compliance Visualization

Now let's build the dashboard charts. We'll use matplotlib (available in any notebook) to visualize the assessment data. In production, you'd use Streamlit + Plotly, but the data transformation is identical.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for notebook
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ===== Chart 1: Compliance Score Gauge =====
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bar chart: status breakdown
statuses = ['PASS', 'FAIL', 'PARTIAL', 'NOT_ASSESSED']
counts = [posture2.passed, posture2.failed, posture2.partial, posture2.not_assessed]
bar_colors = ['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6']

axes[0].bar(statuses, counts, color=bar_colors)
axes[0].set_title(f'Control Status (Score: {posture2.compliance_percentage:.1f}%)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# ===== Chart 2: Family breakdown =====
families = sorted(posture2.by_family.keys())
passed_by_fam = [posture2.by_family[f]['passed'] for f in families]
failed_by_fam = [posture2.by_family[f]['failed'] for f in families]
partial_by_fam = [posture2.by_family[f]['partial'] for f in families]

x = range(len(families))
width = 0.25
axes[1].bar([i - width for i in x], passed_by_fam, width, label='Pass', color='#2ecc71')
axes[1].bar(x, failed_by_fam, width, label='Fail', color='#e74c3c')
axes[1].bar([i + width for i in x], partial_by_fam, width, label='Partial', color='#f39c12')
axes[1].set_xticks(x)
axes[1].set_xticklabels(families, rotation=45)
axes[1].set_title('Status by Family')
axes[1].legend(fontsize=8)

# ===== Chart 3: Compliance trend (scan 1 vs scan 2) =====
scan_labels = ['Jan 8 (Baseline)', 'Jan 15 (Weekly)']
compliance_pcts = [posture1.compliance_percentage, posture2.compliance_percentage]
axes[2].plot(scan_labels, compliance_pcts, 'b-o', linewidth=2, markersize=8)
axes[2].set_ylim(0, 100)
axes[2].set_ylabel('Compliance %')
axes[2].set_title('Compliance Trend')
axes[2].axhline(y=90, color='green', linestyle='--', alpha=0.5, label='Target: 90%')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/dashboard_charts.png', dpi=100, bbox_inches='tight')
plt.show()
print("Charts generated from CompliancePosture data — same objects the API serves")

In [ ]:
# ===== Drift Events Visualization =====
if drift_events:
    fig, ax = plt.subplots(figsize=(10, 3))
    
    for i, event in enumerate(drift_events):
        color = '#e74c3c' if event.is_regression else '#2ecc71'
        marker = 'v' if event.is_regression else '^'
        label = f"{event.control_id}: {event.previous_status} -> {event.current_status}"
        ax.scatter(i, 0, c=color, marker=marker, s=200, zorder=5)
        ax.annotate(label, (i, 0), textcoords="offset points",
                   xytext=(0, 15 if i % 2 == 0 else -20),
                   ha='center', fontsize=8, fontweight='bold')
    
    ax.set_xlim(-0.5, len(drift_events) - 0.5)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title(f'Drift Events: {sum(1 for d in drift_events if d.is_regression)} regressions, '
                 f'{sum(1 for d in drift_events if d.drift_type == "IMPROVEMENT")} improvements')
    
    regression_patch = mpatches.Patch(color='#e74c3c', label='Regression')
    improvement_patch = mpatches.Patch(color='#2ecc71', label='Improvement')
    ax.legend(handles=[regression_patch, improvement_patch], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('/tmp/drift_chart.png', dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("No drift events detected between scans")

In [ ]:
# ===== Remediation Priority Table =====
# This is what the dashboard shows to compliance managers

print(f"{'Priority':>8} {'Control':<10} {'Title':<45} {'Status':<12} {'Severity':<12}")
print("-" * 90)

# Sort by remediation priority (highest first)
sorted_assessments = sorted(assessments2, key=lambda a: a.remediation_priority, reverse=True)
for a in sorted_assessments[:10]:
    if a.status != ControlStatus.NOT_ASSESSED:
        print(f"{a.remediation_priority:>8} {a.control_id:<10} {a.control_title:<45} "
              f"{a.status.value:<12} {a.highest_severity:<12}")

print(f"\nTop failures from posture.top_failures (used by the API):")
for f in posture2.top_failures[:5]:
    print(f"  {f['control_id']}: {f['control_title']} "
          f"(priority {f['remediation_priority']}, {f['failed_findings']} failed findings)")

## Part 3: DDIA Deep Dive — Serving Derived Data (Ch. 12)

### The Read Path vs Write Path

Kleppmann distinguishes between the **write path** (how data enters the system) and the **read path** (how data is served to users). Our system follows this exactly:

**Write path (Phases 1–4):**
```
Collectors → ScanResult → ControlMappingEngine → ControlAssessment[] → DynamoDB
                                                → CompliancePosture    → DynamoDB
                                                → DriftDetector        → DriftEvent[] → DynamoDB
```

**Read path (Phase 5 — this phase):**
```
Dashboard → API Gateway → Lambda → DynamoDB.get_item() → JSON response
```

The write path does the expensive computation (mapping, assessment, drift detection). The read path is a simple lookup — O(1) for DynamoDB point reads, O(n) for scans filtered by family.

**Why this matters (p. 504):** *"If you can define the derived view, you can build a system that keeps it up to date automatically."* Every time a new scan completes, Phase 2 recomputes assessments and Phase 4 detects drift. The dashboard always shows fresh data without doing any computation itself.

### Caching Strategy (System Design Interview Ch. 1)

```
Client → CloudFront (1hr TTL)
           ↓ cache miss
       API Gateway (5min TTL for /posture, 0 for /reports)
           ↓ cache miss
       Lambda → DynamoDB
```

**What to cache:** `/posture` (changes only when a new scan runs, ~daily). `/assessments` (same). `/drift` (same).
**What NOT to cache:** `/reports` (on-demand PDF generation, unique per request).

API Gateway caching is a DVA-C02 exam topic. Key: you set TTL per-method, and cache keys include query parameters by default.

---

## Lab 5.4: Streamlit Dashboard Design

Streamlit turns Python scripts into web dashboards. Here's the architecture:

```
streamlit_app.py
├── Authentication (Cognito login form)
├── Sidebar (filters: family, status, time range)
├── KPI Row (st.metric: total controls, compliance %, failures, drift count)
├── Compliance Trend (st.plotly_chart: time series of compliance %)
├── Family Breakdown (st.plotly_chart: stacked bar by family)
├── Drift Events (st.dataframe: recent regressions/improvements)
└── Control Details (st.dataframe: searchable table of all assessments)
```

The Streamlit app calls our API endpoints and renders the JSON responses as charts and tables. The code below shows the data transformation — the same transformations Streamlit would do, using our real `src/` objects.

In [ ]:
# Simulate what Streamlit would render using our real data
# In production: streamlit run app.py

# KPI metrics (st.metric equivalent)
print("=" * 60)
print("COMPLIANCE DASHBOARD — KPI ROW")
print("=" * 60)
print(f"  Total Controls:   {posture2.total_controls}")
print(f"  Compliance Rate:  {posture2.compliance_percentage:.1f}%")
print(f"  Failed Controls:  {posture2.failed}")
print(f"  Drift Events:     {len(drift_events)} ({sum(1 for d in drift_events if d.is_regression)} regressions)")

# Family breakdown (st.dataframe equivalent)
print(f"\n{'Family':<6} {'Total':>6} {'Pass':>6} {'Fail':>6} {'Partial':>8} {'N/A':>6} {'Rate':>8}")
print("-" * 50)
for fam in sorted(posture2.by_family.keys()):
    c = posture2.by_family[fam]
    rate = (c['passed'] / c['total'] * 100) if c['total'] > 0 else 0
    print(f"  {fam:<4} {c['total']:>6} {c['passed']:>6} {c['failed']:>6} "
          f"{c['partial']:>8} {c.get('not_assessed', 0):>6} {rate:>7.0f}%")

# Drift events table (st.dataframe equivalent)
if drift_events:
    print(f"\nDRIFT EVENTS (between scans)")
    print(f"{'Control':<10} {'Type':<14} {'Previous':<14} {'Current':<14} {'Severity':<10}")
    print("-" * 62)
    for d in drift_events:
        print(f"  {d.control_id:<8} {d.drift_type:<14} {d.previous_status:<14} "
              f"{d.current_status:<14} {d.severity:<10}")

print(f"\nAll data comes from src/models objects — same JSON the API returns")

### Exercise 5.2: Build a Streamlit Dashboard

Create a file `dashboard/app.py` that:
1. Calls the API endpoints (or reads from local data for testing)
2. Shows the KPI row using `st.metric()`
3. Shows the compliance trend using `st.plotly_chart()` (line chart)
4. Shows the family breakdown using `st.plotly_chart()` (stacked bar)
5. Shows the control details using `st.dataframe()` with filtering

**Starter code:**
```python
import streamlit as st
import plotly.graph_objects as go

st.set_page_config(page_title="Compliance Dashboard", layout="wide")
st.title("AWS Compliance Dashboard")

# Load data (replace with API call in production)
# posture = requests.get(f"{API_URL}/posture").json()

col1, col2, col3 = st.columns(3)
col1.metric("Compliance", f"{posture['compliance_percentage']:.1f}%")
col2.metric("Failed", posture['failed'])
col3.metric("Drift Events", len(drift_events))
```

Run with: `streamlit run dashboard/app.py`

### Exercise 5.3: Add Cognito Authentication

Extend the dashboard to:
1. Show a login form in the sidebar (`st.text_input` for email/password)
2. Call Cognito `initiate_auth()` to get JWT tokens
3. Store the token in `st.session_state`
4. Pass the token as `Authorization: Bearer <token>` header on API calls
5. Handle token expiration (refresh when 401 received)

**DVA-C02 exam question:** "How should a web application handle expired Cognito tokens?" → Use the refresh_token to get a new access_token without re-authenticating.

### Exercise 5.4: API Caching Design

Design the caching strategy for each endpoint:

| Endpoint | Cache TTL | Why |
|----------|-----------|-----|
| `GET /posture` | ? | |
| `GET /assessments` | ? | |
| `GET /assessments?family=AC` | ? | |
| `GET /drift` | ? | |
| `POST /reports` | ? | |

Consider: how often does the data change? What's the cost of stale data? What's the cost of a cache miss?

**DDIA connection (Ch. 5):** Kleppmann discusses leader-based replication where followers may serve stale reads. API Gateway caching has the same trade-off — you trade freshness for latency.

### Exercise 5.5: Rate Limiting (System Design Interview Ch. 1)

Design a rate limiting strategy:
1. Unauthenticated requests: 10/min (token bucket)
2. Authenticated requests: 100/min per user
3. Report generation: 5/hour per user (expensive operation)

How would you implement this with API Gateway? What about a custom Lambda authorizer that checks DynamoDB for rate counters?

---

## Lab 5.5: DynamoDB Read Patterns for the API

The API reads from DynamoDB. Here are the access patterns and how they map to our table design (from Phase 4):

```
Access Pattern                          DynamoDB Operation
──────────────                          ──────────────────
Get posture for a scan                  GetItem(PK='POSTURE', SK='SCAN#{scan_id}')
Get all assessments for a scan          Query(GSI1PK='SCAN#{scan_id}', GSI1SK begins_with 'CTRL#')
Get assessment for one control          GetItem(PK='CTRL#{control_id}', SK='SCAN#{scan_id}')
Get control history (all scans)         Query(PK='CTRL#{control_id}', SK begins_with 'SCAN#')
Get drift events for a scan             Query(PK='DRIFT', SK begins_with 'SCAN#{scan_id}')
Get all FAIL controls for a scan        Query(GSI1PK='SCAN#{scan_id}') + FilterExpression status=FAIL
```

**DVA-C02 insight:** The exam tests whether you understand when to use `GetItem` (single item, PK+SK known) vs `Query` (range of items, PK known, SK range) vs `Scan` (full table, avoid in production). Our API uses `GetItem` for posture (fast, O(1)) and `Query` for assessments (efficient, O(n) where n = matching items).

In [ ]:
# Simulate DynamoDB read patterns using our in-memory data
# In production, replace with boto3.resource('dynamodb').Table('...').query(...)

# Pattern 1: Get posture for latest scan
print("=== GetItem: Posture ===")
print(f"  PK: POSTURE, SK: SCAN#{scan2.scan_id}")
print(f"  Result: {posture2.compliance_percentage:.1f}% compliance")

# Pattern 2: Query all FAIL assessments for a scan
print("\n=== Query: Failed Controls ===")
print(f"  GSI1PK: SCAN#{scan2.scan_id}, Filter: status=FAIL")
failed = [a for a in assessments2 if a.status == ControlStatus.FAIL]
for a in failed[:5]:
    print(f"  -> {a.control_id}: {a.control_title} (priority {a.remediation_priority})")

# Pattern 3: Get control history across scans
print("\n=== Query: Control History (AC-2) ===")
print(f"  PK: CTRL#AC-2, SK begins_with SCAN#")
ac2_scan1 = next(a for a in assessments1 if a.control_id == "AC-2")
ac2_scan2 = next(a for a in assessments2 if a.control_id == "AC-2")
print(f"  Scan 1: {ac2_scan1.status.value} (priority {ac2_scan1.remediation_priority})")
print(f"  Scan 2: {ac2_scan2.status.value} (priority {ac2_scan2.remediation_priority})")

# Pattern 4: Get drift events
print("\n=== Query: Drift Events ===")
print(f"  PK: DRIFT, SK begins_with SCAN#{scan2.scan_id}")
for d in drift_events:
    emoji = "REGRESSION" if d.is_regression else "IMPROVEMENT"
    print(f"  -> {d.control_id}: {emoji} ({d.previous_status} -> {d.current_status})")

## Summary

### What you built
- Designed REST API endpoints that serve `ControlAssessment` and `CompliancePosture` as JSON
- Visualized compliance data using matplotlib (same transformations Streamlit uses)
- Explored DynamoDB read patterns for the API
- Built two-scan comparison data with drift detection

### Pipeline so far
```
Phase 1: Collectors → ScanResult
Phase 2: ScanResult → ControlMappingEngine → ControlAssessment[] + CompliancePosture
Phase 3: Assessments + Posture → PDFReportGenerator → PDF → S3
Phase 4: Two scans → DriftDetector → DriftEvent[] → SNS alerts
Phase 5: API Gateway → Lambda → DynamoDB → JSON → Dashboard (THIS PHASE)
Phase 6: (next) → Terraform IaC, CI/CD deployment
```

### Key classes used (all from `src/`)
| Class | Module | API endpoint |
|-------|--------|-------------|
| `CompliancePosture` | `src.models` | `GET /posture` |
| `ControlAssessment` | `src.models` | `GET /assessments` |
| `DriftEvent` | `src.models` | `GET /drift` |
| `PDFReportGenerator` | `src.evidence.pdf_generator` | `POST /reports` |

### DDIA connections
- **Ch. 12 (Derived Data):** Dashboard serves pre-computed views, not raw findings
- **Ch. 1 (Maintainability):** API versioning, backward compatibility, statelessness
- **Ch. 5 (Replication):** Cache staleness trade-off mirrors follower lag

### DVA-C02 connections
- **API Gateway:** REST vs HTTP API, caching, throttling, stage variables (Domain 2)
- **Cognito:** User Pools, JWT tokens, MFA, token lifecycle (Domain 3)
- **Lambda:** Proxy integration response format, cold starts, concurrency (Domain 2)
- **DynamoDB:** GetItem vs Query vs Scan, GSI design for API access patterns (Domain 2)

**Next: Phase 6** — Infrastructure as Code with Terraform, CI/CD with GitHub Actions.